In [13]:
from scipy import stats
import itertools
import glob
import os
import pandas as pd

file_paths = glob.glob("data/*.csv")

monthly_data = {}

for file in file_paths:
    df = pd.read_csv(file)
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.sort_values("timestamp")
    df["dt_seconds"] = df["timestamp"].diff().dt.total_seconds()
    df["energy_kJ"] = df["heating_wattage_W"] * df["dt_seconds"] / 1000
    df["carbon_g"] = df["energy_kJ"] * df["carbon_intensity_gco2_per_kj"]
    df["date"] = df["timestamp"].dt.date
    daily = df.groupby("date")["carbon_g"].sum().reset_index()
    daily["date"] = pd.to_datetime(daily["date"])
    daily["month"] = daily["date"].dt.month

    name = os.path.basename(file)
    monthly_data[name] = {
        m: daily[daily["month"] == m]["carbon_g"].dropna().values
        for m in range(1, 13)
    }

# ── Pairwise comparison per month ─────────────────────────────────────────────
file_names = list(monthly_data.keys())
pairs = list(itertools.combinations(file_names, 2))


for m in range(1, 13):
    print(f"  Month {m}")

    for file in file_names:
        v = monthly_data[file][m]
        _, p = stats.shapiro(v)
        if p <= 0.05:
            print(f"  {file}: {p}")
            print(p > 0.05)



  Month 1
  Month 2
  Month 3
  Month 4
  Month 5
  Month 6
  room_data_bang_c.csv: 0.002847197613409412
False
  room_data_bang_w.csv: 0.014736053237470133
False
  room_data_mpc_c_0.csv: 0.00048470547643689904
False
  room_data_mpc_c_6.csv: 8.547863086406999e-05
False
  room_data_mpc_w_0.csv: 0.017572490385169132
False
  room_data_mpc_w_6.csv: 0.004342621722044617
False
  Month 7
  room_data_mpc_c_0.csv: 0.002219218421046332
False
  room_data_mpc_c_6.csv: 8.264492397071024e-06
False
  Month 8
  room_data_bang_c.csv: 0.0008069996521123041
False
  room_data_bang_w.csv: 0.006937534177964279
False
  room_data_mpc_c_0.csv: 0.00015507467660388956
False
  room_data_mpc_c_6.csv: 1.2455882416307073e-05
False
  room_data_mpc_w_0.csv: 0.010851611331616431
False
  room_data_mpc_w_6.csv: 0.0014492568691528045
False
  Month 9
  room_data_bang_c.csv: 0.0018514047359430284
False
  room_data_bang_w.csv: 0.007743543927554711
False
  room_data_mpc_c_0.csv: 0.001443125585941161
False
  room_data_mpc_c_6.c

In [16]:
from scipy import stats
import itertools
import glob
import os
import pandas as pd

def friedman_per_month(monthly_data: dict) -> pd.DataFrame:
    algorithms = list(monthly_data.keys())
    results = []

    for month in range(1, 13):
        groups = {algo: monthly_data[algo].get(month, []) for algo in algorithms}
        lengths = {algo: len(v) for algo, v in groups.items()}
        n = min(lengths.values())

        if n < 2:
            results.append({
                "month": month,
                "n_days": n,
                "statistic": None,
                "p_value": None,
                "significant": None,
                "note": "insufficient data",
            })
            continue

        trimmed = [groups[algo][:n] for algo in algorithms]
        any_trimmed = len(set(lengths.values())) > 1

        try:
            stat, p = stats.friedmanchisquare(*trimmed)
            results.append({
                "month": month,
                "n_days": n,
                "statistic": round(stat, 4),
                "p_value": round(p, 4),
                "significant": p < 0.05,
                "note": "trimmed to shortest" if any_trimmed else "exact",
            })
        except ValueError as e:
            results.append({
                "month": month,
                "n_days": n,
                "statistic": None,
                "p_value": None,
                "significant": None,
                "note": str(e),
            })

    return pd.DataFrame(results)


def posthoc_wilcoxon_per_month(monthly_data: dict, friedman_results: pd.DataFrame) -> pd.DataFrame:
    from statsmodels.stats.multitest import multipletests

    algorithms = list(monthly_data.keys())
    pairs = list(itertools.combinations(algorithms, 2))
    sig_months = friedman_results[friedman_results["significant"] == True]["month"].tolist()
    results = []

    for month in sig_months:
        groups = {algo: monthly_data[algo].get(month, []) for algo in algorithms}
        n = min(len(v) for v in groups.values())
        pair_results = []

        for algo_1, algo_2 in pairs:
            a, b = groups[algo_1][:n], groups[algo_2][:n]
            try:
                stat, p = stats.wilcoxon(a, b)
                pair_results.append({
                    "month": month,
                    "algo_1": algo_1,
                    "algo_2": algo_2,
                    "statistic": round(stat, 4),
                    "p_value": round(p, 4),
                })
            except ValueError as e:
                pair_results.append({
                    "month": month,
                    "algo_1": algo_1,
                    "algo_2": algo_2,
                    "statistic": None,
                    "p_value": None,
                })

        # Bonferroni correction across all pairs within this month
        pair_df = pd.DataFrame(pair_results)
        valid = pair_df["p_value"].notna()
        if valid.any():
            _, p_corr, _, _ = multipletests(pair_df.loc[valid, "p_value"], method="bonferroni")
            pair_df.loc[valid, "p_bonferroni"] = p_corr.round(4)
            pair_df.loc[valid, "significant_bonferroni"] = p_corr < 0.05

        results.append(pair_df)

    return pd.concat(results, ignore_index=True) if results else pd.DataFrame()


# --- Run ---
friedman_df = friedman_per_month(monthly_data)
print("=== Friedman Test per Month ===")
print(friedman_df.to_string(index=False))

posthoc_df = posthoc_wilcoxon_per_month(monthly_data, friedman_df)
if not posthoc_df.empty:
    print("\n=== Post-hoc Wilcoxon (significant months only) ===")
    print(posthoc_df.to_string(index=False))
else:
    print("\nNo significant months — no post-hoc tests run.")

=== Friedman Test per Month ===
 month  n_days  statistic  p_value  significant  note
     1      32   148.1091      0.0         True exact
     2      28   133.4490      0.0         True exact
     3      31   145.1751      0.0         True exact
     4      30   132.1905      0.0         True exact
     5      31   128.9130      0.0         True exact
     6      30   115.6187      0.0         True exact
     7      31   135.8661      0.0         True exact
     8      31   132.0485      0.0         True exact
     9      30   142.6746      0.0         True exact
    10      31   152.1982      0.0         True exact
    11      30   145.2952      0.0         True exact
    12      31   149.0276      0.0         True exact

=== Post-hoc Wilcoxon (significant months only) ===
 month                algo_1                algo_2  statistic  p_value  p_bonferroni significant_bonferroni
     1  room_data_bang_c.csv  room_data_bang_w.csv        0.0   0.0000        0.0000                   Tr

In [ ]:
from scipy import stats
import itertools
import glob
import os
import pandas as pd
import numpy as np

file_paths = glob.glob("data/*_c*.csv")
monthly_data = {}

for file in file_paths:
    df = pd.read_csv(file)
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.sort_values("timestamp")
    df["dt_seconds"] = df["timestamp"].diff().dt.total_seconds()
    df["energy_kJ"] = df["heating_wattage_W"] * df["dt_seconds"] / 1000
    df["carbon_g"] = df["energy_kJ"] * df["carbon_intensity_gco2_per_kj"]
    df["date"] = df["timestamp"].dt.date
    daily = df.groupby("date")["carbon_g"].sum().reset_index()
    daily["date"] = pd.to_datetime(daily["date"])
    daily["month"] = daily["date"].dt.month
    name = os.path.basename(file)
    monthly_data[name] = {
        m: daily[daily["month"] == m][["date", "carbon_g"]].reset_index(drop=True)
        for m in range(1, 13)
    }


def compute_residuals_per_month(monthly_data: dict) -> dict:
    """
    For each month, aligns all algorithms on shared dates, then subtracts
    the per-day mean across algorithms to produce day-demeaned residuals.

    Only days present in ALL 6 algorithms are kept (complete blocks).

    Args:
        monthly_data: dict mapping filename -> {month: DataFrame with date, carbon_g}

    Returns:
        dict mapping month -> DataFrame with columns: date, algo_1, ..., algo_6 (residuals)
    """
    algorithms = list(monthly_data.keys())
    residuals_by_month = {}

    for month in range(1, 13):
        # Align on shared dates via inner join
        merged = None
        for algo in algorithms:
            df = monthly_data[algo][month][["date", "carbon_g"]].rename(
                columns={"carbon_g": algo}
            )
            merged = df if merged is None else merged.merge(df, on="date", how="inner")

        if merged is None or len(merged) < 2:
            residuals_by_month[month] = None
            continue

        # Subtract per-day mean across all algorithms
        algo_cols = algorithms
        day_means = merged[algo_cols].mean(axis=1)
        residuals = merged[algo_cols].subtract(day_means, axis=0)
        residuals.insert(0, "date", merged["date"])
        residuals_by_month[month] = residuals

    return residuals_by_month


def friedman_on_residuals(residuals_by_month: dict) -> pd.DataFrame:
    """
    Runs Friedman test per month on day-demeaned residuals.
    Rows = days (blocks), columns = algorithms (treatments).

    Args:
        residuals_by_month: output of compute_residuals_per_month()

    Returns:
        DataFrame with columns: month, n_days, statistic, p_value, significant
    """
    results = []

    for month, res_df in residuals_by_month.items():
        if res_df is None:
            results.append({
                "month": month, "n_days": 0,
                "statistic": None, "p_value": None,
                "significant": None, "note": "insufficient data",
            })
            continue

        algo_cols = [c for c in res_df.columns if c != "date"]
        groups = [res_df[col].values for col in algo_cols]

        try:
            stat, p = stats.friedmanchisquare(*groups)
            results.append({
                "month": month,
                "n_days": len(res_df),
                "statistic": round(stat, 4),
                "p_value": round(p, 4),
                "significant": p < 0.05,
                "note": "ok",
            })
        except ValueError as e:
            results.append({
                "month": month, "n_days": len(res_df),
                "statistic": None, "p_value": None,
                "significant": None, "note": str(e),
            })

    return pd.DataFrame(results)


def posthoc_wilcoxon_on_residuals(
    residuals_by_month: dict, friedman_results: pd.DataFrame
) -> pd.DataFrame:
    """
    For significant months, runs pairwise Wilcoxon signed-rank tests on residuals
    across all algorithm pairs, with Bonferroni correction within each month.

    Args:
        residuals_by_month: output of compute_residuals_per_month()
        friedman_results: output of friedman_on_residuals()

    Returns:
        DataFrame with columns:
            month, algo_1, algo_2, statistic, p_value, p_bonferroni, significant_bonferroni
    """
    from statsmodels.stats.multitest import multipletests

    sig_months = friedman_results[friedman_results["significant"] == True]["month"].tolist()
    all_results = []

    for month in sig_months:
        res_df = residuals_by_month[month]
        algo_cols = [c for c in res_df.columns if c != "date"]
        pairs = list(itertools.combinations(algo_cols, 2))
        pair_results = []

        for algo_1, algo_2 in pairs:
            a = res_df[algo_1].values
            b = res_df[algo_2].values
            try:
                stat, p = stats.wilcoxon(a, b)
                pair_results.append({
                    "month": month, "algo_1": algo_1, "algo_2": algo_2,
                    "statistic": round(stat, 4), "p_value": round(p, 4),
                })
            except ValueError as e:
                pair_results.append({
                    "month": month, "algo_1": algo_1, "algo_2": algo_2,
                    "statistic": None, "p_value": None,
                })

        pair_df = pd.DataFrame(pair_results)
        valid = pair_df["p_value"].notna()
        if valid.any():
            _, p_corr, _, _ = multipletests(pair_df.loc[valid, "p_value"], method="bonferroni")
            pair_df.loc[valid, "p_bonferroni"] = p_corr.round(4)
            pair_df.loc[valid, "significant_bonferroni"] = p_corr < 0.05

        all_results.append(pair_df)

    return pd.concat(all_results, ignore_index=True) if all_results else pd.DataFrame()


# --- Run ---
residuals_by_month = compute_residuals_per_month(monthly_data)

friedman_df = friedman_on_residuals(residuals_by_month)
print("=== Friedman Test on Residuals per Month ===")
print(friedman_df.to_string(index=False))

posthoc_df = posthoc_wilcoxon_on_residuals(residuals_by_month, friedman_df)
if not posthoc_df.empty:
    print("\n=== Post-hoc Wilcoxon on Residuals (significant months only) ===")
    print(posthoc_df.to_string(index=False))
else:
    print("\nNo significant months — no post-hoc tests run.")

=== Friedman Test on Residuals per Month ===
 month  n_days  statistic  p_value  significant note
     1      32   148.1091      0.0         True   ok
     2      28   133.4490      0.0         True   ok
     3      31   145.1751      0.0         True   ok
     4      30   132.1905      0.0         True   ok
     5      31   128.9130      0.0         True   ok
     6      30   115.6187      0.0         True   ok
     7      31   135.8661      0.0         True   ok
     8      31   132.0485      0.0         True   ok
     9      30   142.6746      0.0         True   ok
    10      31   152.1982      0.0         True   ok
    11      30   145.2952      0.0         True   ok
    12      31   149.0276      0.0         True   ok

=== Post-hoc Wilcoxon on Residuals (significant months only) ===
 month                algo_1                algo_2  statistic  p_value  p_bonferroni significant_bonferroni
     1  room_data_bang_c.csv  room_data_bang_w.csv        0.0   0.0000        0.0000        